# 🕵️ Dia 28 — Forense de metadados, parte 2: PDFs e documentos Office

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 28: Metadata forensics, part 2: PDFs and Office documents* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-28.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Quinta, 8 de outubro. O `Comunicado_Oficial_SerraAzul.pdf` circula nos grupos com o logo da cooperativa e o link `serraazul-pix.example`. Rafa rodou `grep -ac '%%EOF'` e a tela mostrou `3`.

*"Parece de vocês porque **é** de vocês. Ou foi, na primeira vez que alguém salvou."*

Neste notebook você vai:

1. **Contar revisões** e ler o dicionário Info de cada uma, só com `re`.
2. **Recortar a revisão 1** e ler o texto que o golpista sobrescreveu.
3. **Abrir um `.docx` por dentro** com `zipfile` + `xml.etree`.
4. **Montar o mapa de calor software × ano** sem colunas de nomes.
5. **Jogar o quiz "normal ou anomalia?"**.

🐍 Só biblioteca padrão. Tudo fictício.

## 1. Carregar o PDF, com hash primeiro

Plano A: baixar do GitHub. Plano B: a cópia embutida (o arquivo tem 2,6 KB).

In [ ]:
import base64, hashlib, urllib.request

URL = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/"
       "main/caso-continuo/dados/dia-28/Comunicado_Oficial_SerraAzul.pdf")
EMBUTIDO = (
    "JVBERi0xLjQKJeLjz9MKMSAwIG9iago8PCAvVHlwZSAvQ2F0YWxvZyAvUGFnZXMgMiAwIFIgPj4KZW5kb2JqCjIgMCBvYmoKPDwg"
    "L1R5cGUgL1BhZ2VzIC9LaWRzIFszIDAgUl0gL0NvdW50IDEgPj4KZW5kb2JqCjMgMCBvYmoKPDwgL1R5cGUgL1BhZ2UgL1BhcmVu"
    "dCAyIDAgUiAvTWVkaWFCb3ggWzAgMCA1OTUgODQyXSAvQ29udGVudHMgNCAwIFIgL1Jlc291cmNlcyA8PCAvRm9udCA8PCAvRjEg"
    "NSAwIFIgPj4gPj4gPj4KZW5kb2JqCjQgMCBvYmoKPDwgL0xlbmd0aCAzNDEgPj4Kc3RyZWFtCkJUCi9GMSAxMyBUZgo1MCA3ODAg"
    "VGQKMTYgVEwKKENPT1BFUkFUSVZBIFNFUlJBIEFaVUwpIFRqIFQqCihDb211bmljYWRvIGFvcyBjb29wZXJhZG9zKSBUaiBUKgoo"
    "KSBUaiBUKgooSG9yXDM0MXJpbyBkZSBhdGVuZGltZW50byBkZSBmaW0gZGUgYW5vOikgVGogVCoKKDI0LzEyIGUgMzEvMTIgLSBh"
    "Z1wzNTJuY2lhcyBhYmVydGFzIGF0XDM1MSAxMmguKSBUaiBUKgooUGl4IGUgYXBwIGZ1bmNpb25hbSBub3JtYWxtZW50ZS4pIFRq"
    "IFQqCigpIFRqIFQqCihBIFNlcnJhIEF6dWwgbnVuY2EgcGVkZSBzZW5oYSBwb3IgU01TIG91IGxpbmsuKSBUaiBUKgooc2VycmFh"
    "enVsLmV4YW1wbGUpIFRqIFQqCkVUCmVuZHN0cmVhbQplbmRvYmoKNSAwIG9iago8PCAvVHlwZSAvRm9udCAvU3VidHlwZSAvVHlw"
    "ZTEgL0Jhc2VGb250IC9IZWx2ZXRpY2EgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgPj4KZW5kb2JqCjYgMCBvYmoKPDwgL1Rp"
    "dGxlIChDb211bmljYWRvIC0gaG9yYXJpbyBkZSBmaW0gZGUgYW5vKSAvQXV0aG9yIChDb211bmljYWNhbyBTZXJyYSBBenVsKSAv"
    "Q3JlYXRvciAoV3JpdGVyKSAvUHJvZHVjZXIgKExpYnJlT2ZmaWNlIDcuNikgL0NyZWF0aW9uRGF0ZSAoRDoyMDI1MTIwMTA5MzAw"
    "MC0wMycwMCcpIC9Nb2REYXRlIChEOjIwMjUxMjAxMDkzMDAwLTAzJzAwJykgPj4KZW5kb2JqCnhyZWYKMCAxCjAwMDAwMDAwMDAg"
    "NjU1MzUgZiAKMSA2CjAwMDAwMDAwMTUgMDAwMDAgbiAKMDAwMDAwMDA2NCAwMDAwMCBuIAowMDAwMDAwMTIxIDAwMDAwIG4gCjAw"
    "MDAwMDAyNDcgMDAwMDAgbiAKMDAwMDAwMDYzOSAwMDAwMCBuIAowMDAwMDAwNzM2IDAwMDAwIG4gCnRyYWlsZXIKPDwgL1NpemUg"
    "NyAvUm9vdCAxIDAgUiAvSW5mbyA2IDAgUiAvSUQgWzw1QTdFMUMwRjJCOUQ0RTZBOEMzRjFCMkQ0RTZGOEE5MD4gPDVBN0UxQzBG"
    "MkI5RDRFNkE4QzNGMUIyRDRFNkY4QTkwPl0gPj4Kc3RhcnR4cmVmCjk1NQolJUVPRgozIDAgb2JqCjw8IC9UeXBlIC9QYWdlIC9Q"
    "YXJlbnQgMiAwIFIgL01lZGlhQm94IFswIDAgNTk1IDg0Ml0gL0NvbnRlbnRzIDcgMCBSIC9SZXNvdXJjZXMgPDwgL0ZvbnQgPDwg"
    "L0YxIDUgMCBSID4+ID4+ID4+CmVuZG9iago3IDAgb2JqCjw8IC9MZW5ndGggMzQwID4+CnN0cmVhbQpCVAovRjEgMTMgVGYKNTAg"
    "NzgwIFRkCjE2IFRMCihDT09QRVJBVElWQSBTRVJSQSBBWlVMKSBUaiBUKgooQ29tdW5pY2FkbyBhb3MgY29vcGVyYWRvcykgVGog"
    "VCoKKCkgVGogVCoKKFBvciBzZWd1cmFuXDM0N2EsIGNoYXZlcyBQaXggc2VtIHJlY2FkYXN0cm8pIFRqIFQqCihzZXJcMzQzbyBC"
    "TE9RVUVBREFTIGVtIDI0IGhvcmFzLikgVGogVCoKKFJlZ3VsYXJpemUgYWdvcmEgZW0gc2VycmFhenVsLXBpeC5leGFtcGxlKSBU"
    "aiBUKgooKSBUaiBUKgooQXRlbmRpbWVudG86IFNlcnJhIEF6dWwgQXRlbmRpbWVudG8gXChXaGF0c0FwcFwpKSBUaiBUKgooc2Vy"
    "cmFhenVsLmV4YW1wbGUpIFRqIFQqCkVUCmVuZHN0cmVhbQplbmRvYmoKOCAwIG9iago8PCAvVGl0bGUgKENvbXVuaWNhZG8gLSBo"
    "b3JhcmlvIGRlIGZpbSBkZSBhbm8pIC9BdXRob3IgKENvbXVuaWNhY2FvIFNlcnJhIEF6dWwpIC9DcmVhdG9yIChXcml0ZXIpIC9Q"
    "cm9kdWNlciAoUERGUmFwaWRvIE9ubGluZSA0LjApIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNTEyMDEwOTMwMDAtMDMnMDAnKSAvTW9k"
    "RGF0ZSAoRDoyMDI2MTAwNDIyNDgwMC0wMycwMCcpID4+CmVuZG9iagp4cmVmCjMgMQowMDAwMDAxMjUwIDAwMDAwIG4gCjcgMgow"
    "MDAwMDAxMzc2IDAwMDAwIG4gCjAwMDAwMDE3NjcgMDAwMDAgbiAKdHJhaWxlcgo8PCAvU2l6ZSA5IC9Sb290IDEgMCBSIC9JbmZv"
    "IDggMCBSIC9JRCBbPDVBN0UxQzBGMkI5RDRFNkE4QzNGMUIyRDRFNkY4QTkwPiA8OUYxRTJEM0M0QjVBNjk3ODg3OTZBNUI0QzNE"
    "MkUxRjA+XSAvUHJldiA5NTUgPj4Kc3RhcnR4cmVmCjE5OTEKJSVFT0YKOSAwIG9iago8PCAvVGl0bGUgKENvbXVuaWNhZG8gT2Zp"
    "Y2lhbCAtIFJlZ3VsYXJpemFjYW8gUGl4KSAvQXV0aG9yIChDb211bmljYWNhbyBTZXJyYSBBenVsKSAvQ3JlYXRvciAoV3JpdGVy"
    "KSAvUHJvZHVjZXIgKFBERlJhcGlkbyBPbmxpbmUgNC4wKSAvQ3JlYXRpb25EYXRlIChEOjIwMjUxMjAxMDkzMDAwLTAzJzAwJykg"
    "L01vZERhdGUgKEQ6MjAyNjEwMDQyMzE1MDAtMDMnMDAnKSA+PgplbmRvYmoKeHJlZgo5IDEKMDAwMDAwMjIxNyAwMDAwMCBuIAp0"
    "cmFpbGVyCjw8IC9TaXplIDEwIC9Sb290IDEgMCBSIC9JbmZvIDkgMCBSIC9JRCBbPDVBN0UxQzBGMkI5RDRFNkE4QzNGMUIyRDRF"
    "NkY4QTkwPiA8MEExQjJDM0Q0RTVGNjA3MTgyOTNBNEI1QzZEN0U4Rjk+XSAvUHJldiAxOTkxID4+CnN0YXJ0eHJlZgoyNDQ1CiUl"
    "RU9GCg=="
)

try:
    with urllib.request.urlopen(URL, timeout=10) as r:   # plano A: rede
        pdf = r.read()
    origem = "GitHub"
except Exception as e:                                    # plano B: sem rede
    pdf = base64.b64decode(EMBUTIDO)
    origem = f"cópia embutida ({type(e).__name__})"

print("Origem:", origem, "|", len(pdf), "bytes")
print("SHA-256:", hashlib.sha256(pdf).hexdigest())     # sempre antes de analisar
open("comunicado.pdf", "wb").write(pdf)                  # cópia de trabalho

## 2. Quantas vezes foi salvo?

É o Passo 3 do original (`grep -ac '%%EOF'`), em Python. Aproveitamos para ver **onde** cada `%%EOF` está e se o PDF é linearizado (o que explicaria um `%%EOF` a mais sem edição).

In [ ]:
import re

posicoes = [m.start() for m in re.finditer(rb"%%EOF", pdf)]   # = grep -abo '%%EOF'
linearizado = b"/Linearized" in pdf[:1024]                       # o marcador fica no começo do arquivo

print("Contagem de %%EOF:", len(posicoes), "nas posições", posicoes)
print("Linearizado?", "sim" if linearizado else "não")
if len(posicoes) >= 3 or (len(posicoes) == 2 and not linearizado):
    print("➡️ Atualizações incrementais: o arquivo foi salvo mais de uma vez. Vale um olhar.")

## 3. O Info de cada revisão

Cada atualização incremental grava um dicionário Info **novo** no fim do arquivo. O exiftool e o pdfinfo mostram só o último. Aqui listamos todos, na ordem em que foram gravados, e dizemos a qual revisão cada um pertence.

In [ ]:
from datetime import datetime, timedelta, timezone


def data_pdf(s: str):
    """Converte 'D:20261004224800-03'00'' em datetime com fuso."""
    m = re.match(r"D:(\d{14})([+-])(\d{2})'(\d{2})", s)
    base = datetime.strptime(m.group(1), "%Y%m%d%H%M%S")
    sinal = 1 if m.group(2) == "+" else -1
    return base.replace(tzinfo=timezone(sinal * timedelta(hours=int(m.group(3)), minutes=int(m.group(4)))))


def campo(dicionario: bytes, nome: str) -> str:
    """Pega o valor (texto entre parênteses) de um campo do Info."""
    m = re.search(rb"/" + nome.encode() + rb" \((.*?)\)", dicionario)
    return m.group(1).decode("latin-1") if m else "-"


infos = []
for m in re.finditer(rb"<< /Title .*?>>", pdf, re.S):          # cada dicionário Info
    revisao = 1 + sum(1 for p in posicoes if p < m.start())       # quantos %%EOF vieram antes
    d = m.group(0)
    infos.append({
        "revisao": revisao,
        "Title": campo(d, "Title"), "Producer": campo(d, "Producer"),
        "CreationDate": data_pdf(campo(d, "CreationDate")),
        "ModDate": data_pdf(campo(d, "ModDate")),
    })

for i in infos:
    print(f"Revisão {i['revisao']}: Producer={i['Producer']!r}")
    print(f"    Title={i['Title']!r}")
    print(f"    CreationDate={i['CreationDate'].isoformat()}  ModDate={i['ModDate'].isoformat()}")
    print(f"    ModDate em UTC: {i['ModDate'].astimezone(timezone.utc):%Y-%m-%d %H:%M} UTC")

> 🔍 Repare: o `CreationDate` é o mesmo nas três revisões (dezembro de 2025). Ele **herdou** a data do documento original. Quem olhar só o CreationDate acha que o comunicado é antigo e legítimo. O `Producer` e o `ModDate` contam o resto.

## 4. Recortar a revisão 1 e ler o texto apagado

O PDF original termina no primeiro `%%EOF`. Cortando o arquivo ali, temos um PDF válido com o conteúdo de antes da edição. Para ler o texto sem biblioteca, pegamos o fluxo de conteúdo (`stream ... endstream`) e extraímos o que está entre parênteses antes de `Tj` (o operador que desenha texto).

In [ ]:
rev1 = pdf[:posicoes[0] + len(b"%%EOF") + 1]                 # = head -c <posição + 6>
open("rev1.pdf", "wb").write(rev1)
print("rev1.pdf:", len(rev1), "bytes | SHA-256:", hashlib.sha256(rev1).hexdigest()[:16], "...")


def texto_dos_streams(trecho: bytes):
    """Extrai as strings desenhadas com Tj em cada stream não comprimido."""
    saida = []
    for s in re.finditer(rb"stream\n(.*?)\nendstream", trecho, re.S):
        linhas = re.findall(rb"\((.*?)\) Tj", s.group(1))
        # \ooo é um caractere em octal (acentos); decodificamos para latin-1
        texto = [re.sub(rb"\\(\d{3})", lambda x: bytes([int(x.group(1), 8)]), l).decode("latin-1")
                 .replace("\\(", "(").replace("\\)", ")")      # parênteses escapados no PDF
                 for l in linhas]
        saida.append(texto)
    return saida

print("\n=== Texto da revisão 1 (antes da edição) ===")
for linha in texto_dos_streams(rev1)[0]:
    print("  ", linha)

print("\n=== Texto que a última revisão mostra ===")
for linha in texto_dos_streams(pdf)[-1]:
    print("  ", linha)

In [ ]:
# OPCIONAL: as mesmas leituras com as ferramentas do original (o teste pula linhas com !)
!apt-get -qq install -y poppler-utils libimage-exiftool-perl > /dev/null
!pdfinfo comunicado.pdf
!pdfinfo rev1.pdf
!exiftool -a -G1 -s comunicado.pdf
print("(fora do Colab, as linhas com ! foram puladas)")

## 5. Office por dentro: a caixa vermelha do diagrama

Um `.docx` é um ZIP. Vamos **criar** um mínimo (fictício) e depois lê-lo como faria com um documento coletado. Assim você vê exatamente onde ficam os nomes de pessoas que o Passo 5 manda retirar.

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

CORE = """<?xml version="1.0" encoding="UTF-8"?>
<cp:coreProperties xmlns:cp="http://schemas.openxmlformats.org/package/2006/metadata/core-properties"
 xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dcterms="http://purl.org/dc/terms/"
 xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
 <dc:creator>Fulana de Tal (fictícia)</dc:creator>
 <cp:lastModifiedBy>Beltrano (fictício)</cp:lastModifiedBy>
 <cp:revision>14</cp:revision>
 <dcterms:created xsi:type="dcterms:W3CDTF">2019-11-04T15:22:00Z</dcterms:created>
 <dcterms:modified xsi:type="dcterms:W3CDTF">2025-02-10T09:03:00Z</dcterms:modified>
</cp:coreProperties>"""
APP = """<?xml version="1.0" encoding="UTF-8"?>
<Properties xmlns="http://schemas.openxmlformats.org/officeDocument/2006/extended-properties">
 <Template>Modelo-Comunicado-SerraAzul.dotx</Template><TotalTime>37</TotalTime>
 <Application>Microsoft Office Word</Application><AppVersion>16.0000</AppVersion>
 <Company>Cooperativa Serra Azul (fictícia)</Company>
</Properties>"""

with zipfile.ZipFile("tarifas-2025.docx", "w") as z:          # um docx mínimo, só com metadados
    z.writestr("docProps/core.xml", CORE)
    z.writestr("docProps/app.xml", APP)

# --- agora, como investigador: = unzip -p arquivo.docx docProps/core.xml
with zipfile.ZipFile("tarifas-2025.docx") as z:
    print("Conteúdo do ZIP:", z.namelist())
    core = ET.fromstring(z.read("docProps/core.xml"))
    app = ET.fromstring(z.read("docProps/app.xml"))

for elem in list(core) + list(app):
    nome = elem.tag.split("}")[-1]                         # tira o namespace {http://...}
    pessoal = nome in ("creator", "lastModifiedBy")        # a caixa vermelha
    print(f"{'🟥' if pessoal else '  '} {nome:16s} {elem.text}")
print("\n🟥 = nome de pessoa: fica fora do meta.csv. Created 2019 / modified 2025 = linhagem de modelo.")

## 6. O mapa de calor da casa

Trinta documentos públicos (fictícios) de `serraazul.example`, como se você tivesse feito os Passos 1 e 5. A coluna de autor já vem **descartada** na leitura. Depois, colocamos o comunicado na tabela.

In [ ]:
import csv, io
from collections import Counter

# meta.csv fictício: arquivo, Producer, CreateDate, Author (que vamos descartar)
META = """FileName,Producer,CreateDate,Author
relatorio-2021.pdf,Acrobat Distiller 11.0,2021:03:10,Pessoa A
tarifas-2021.pdf,Acrobat Distiller 11.0,2021:01:05,Pessoa B
assembleia-2021.pdf,Acrobat Distiller 11.0,2021:04:22,Pessoa A
edital-2021.pdf,Canon iR-ADV Scan,2021:06:30,
ata-2021.pdf,Canon iR-ADV Scan,2021:07:15,
balanco-2021.pdf,Acrobat Distiller 11.0,2021:12:01,Pessoa C
relatorio-2022.pdf,Acrobat Distiller 11.0,2022:03:11,Pessoa A
tarifas-2022.pdf,LibreOffice 7.2,2022:01:04,Pessoa B
assembleia-2022.pdf,LibreOffice 7.2,2022:04:20,Pessoa A
edital-2022.pdf,Canon iR-ADV Scan,2022:06:28,
balanco-2022.pdf,LibreOffice 7.3,2022:12:02,Pessoa C
seguranca-2022.pdf,LibreOffice 7.3,2022:09:09,Pessoa D
relatorio-2023.pdf,LibreOffice 7.5,2023:03:09,Pessoa A
tarifas-2023.pdf,LibreOffice 7.5,2023:01:03,Pessoa B
assembleia-2023.pdf,LibreOffice 7.5,2023:04:19,Pessoa A
balanco-2023.pdf,LibreOffice 7.5,2023:12:04,Pessoa C
seguranca-2023.pdf,LibreOffice 7.5,2023:09:12,Pessoa D
natal-2023.pdf,LibreOffice 7.5,2023:12:01,Pessoa E
relatorio-2024.pdf,LibreOffice 7.6,2024:03:12,Pessoa A
tarifas-2024.pdf,LibreOffice 7.6,2024:01:08,Pessoa B
assembleia-2024.pdf,LibreOffice 7.6,2024:04:18,Pessoa A
balanco-2024.pdf,LibreOffice 7.6,2024:12:03,Pessoa C
seguranca-2024.pdf,LibreOffice 7.6,2024:09:10,Pessoa D
natal-2024.pdf,LibreOffice 7.6,2024:12:02,Pessoa E
relatorio-2025.pdf,LibreOffice 7.6,2025:03:11,Pessoa A
tarifas-2025.pdf,LibreOffice 7.6,2025:01:07,Pessoa B
assembleia-2025.pdf,LibreOffice 7.6,2025:04:17,Pessoa A
balanco-2025.pdf,LibreOffice 7.6,2025:12:05,Pessoa C
seguranca-2025.pdf,LibreOffice 7.6,2025:09:15,Pessoa D
natal-2025.pdf,LibreOffice 7.6,2025:12:01,Pessoa E
"""

linhas = []
for r in csv.DictReader(io.StringIO(META)):
    r.pop("Author")                                    # minimização: o padrão não precisa de nomes
    linhas.append(r)


def familia(producer: str) -> str:
    """Agrupa strings brutas de Producer em famílias (Passo 6)."""
    p = producer.lower()
    if "acrobat" in p: return "Acrobat"
    if "libreoffice" in p: return "LibreOffice"
    if "scan" in p: return "scanner"
    if "online" in p: return "serviço PDF online"
    return "outro"


def mapa_de_calor(registros):
    anos = sorted({r["CreateDate"][:4] for r in registros})
    cont = Counter((familia(r["Producer"]), r["CreateDate"][:4]) for r in registros)
    familias = sorted({f for f, _ in cont})
    blocos = " ░▒▓█"                                    # escala de "cor" em texto
    print(f"{'SoftwareFamily':20s}" + "".join(f"{a:>7s}" for a in anos))
    for f in familias:
        celulas = ""
        for a in anos:
            n = cont[(f, a)]
            celulas += f"{blocos[min(n, 4)]}{n:>5d} "
        print(f"{f:20s}{celulas}")
    print(f"\nLegenda: {len(registros)} arquivos; campos com nomes de pessoas removidos.")

mapa_de_calor(linhas)

In [ ]:
# Agora o "comunicado" entra na tabela como se fosse da casa (Producer da última revisão).
ultimo = infos[-1]
linhas_com_comunicado = linhas + [{
    "FileName": "Comunicado_Oficial_SerraAzul.pdf",
    "Producer": ultimo["Producer"],
    "CreateDate": ultimo["CreationDate"].strftime("%Y:%m:%d"),
}]
mapa_de_calor(linhas_com_comunicado)
print("\n➡️ Nenhum documento da cooperativa saiu de um 'serviço PDF online'. É o ponto fora da curva.")
print("   E o CreateDate (2025) é herdado: a data real de edição está no ModDate da revisão 2.")

## 7. 🎮 Quiz: normal ou anomalia?

Responda **S** se for uma anomalia que merece entrar no registro, **N** se for normal/esperado. `MODO_JOGO = False` mostra só o gabarito.

In [ ]:
QUIZ = [
    ("PDF linearizado com 2 marcadores %%EOF.", "N",
     "Linearização costuma gerar 2 mesmo com um único salvamento."),
    ("PDF não linearizado com 3 marcadores %%EOF.", "S",
     "Três salvamentos: vale recortar e ler as revisões."),
    ("Creator = Writer e Producer = LibreOffice 7.6, igual aos outros documentos da casa.", "N",
     "Bate com a linha de base da organização."),
    ("Producer de 'serviço PDF online' num comunicado institucional.", "S",
     "Fora da linha de base da casa; é o indicador de falsificação do original."),
    ("CreationDate de dezembro de 2025 com ModDate de outubro de 2026, na véspera de um golpe.", "S",
     "Documento antigo reaproveitado e editado: linhagem + data de edição suspeita."),
    ("docx criado em 2019 e publicado como tarifas de 2025.", "N",
     "Linhagem de modelo: muitas vezes inofensiva. Registre, mas não é achado sozinho."),
    ("Author = 'Comunicacao Serra Azul' prova que a cooperativa fez o comunicado.", "N",
     "O campo foi herdado da revisão 1 (o documento verdadeiro). Qualquer um edita metadado."),
    ("exiftool e pdfinfo discordam do Producer no mesmo arquivo.", "S",
     "Sinal de revisões com Info diferentes. Leia cada revisão."),
]

MODO_JOGO = True
acertos = 0
for i, (frase, certo, porque) in enumerate(QUIZ, 1):
    print(f"\n[{i}/{len(QUIZ)}] {frase}")
    if MODO_JOGO:
        r = ""
        while r not in ("S", "N"):
            r = input("   Anomalia? (S/N): ").strip().upper()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌", end=" ")
    print(f"Gabarito: {certo}. {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)} — meta do Checkpoint BR: 6.")

## 8. 📝 O achado em uma frase

Preencha e gere a linha do registro de reconhecimento (Passo 7 do original).

In [ ]:
#@title Linha de anomalia { display-mode: "form" }
achado = "O comunicado é o PDF de fim de ano da Serra Azul (LibreOffice, 01/12/2025) reeditado duas vezes num serviço PDF online em 04/10/2026, véspera do disparo dos SMS."  #@param {type:"string"}
confianca = "confirmado"  #@param ["confirmado", "provável", "contestado"]
fontes = "revisões 1-3 lidas por re (notebook) e pdfinfo; texto da revisão 1 confere com o site serraazul.example (Marta)"  #@param {type:"string"}

registro = (f"| Comunicado_Oficial_SerraAzul.pdf | {achado} | {confianca} | {fontes} | "
            f"SHA-256 {hashlib.sha256(pdf).hexdigest()[:16]}... |")
print(registro)
open("anomalias.md", "a", encoding="utf-8").write(registro + "\n")
print("\n💾 anexado a anomalias.md")
if confianca == "confirmado":
    print("Confirmado exige 2 fontes independentes: aqui, a estrutura do arquivo E a confirmação do site.")

## ✅ Antes de fechar o Dia 28

- [ ] Três revisões identificadas e o Info de cada uma anotado
- [ ] `rev1.pdf` recortado, com hash, e o texto original lido
- [ ] Mapa de calor com legenda (nº de arquivos + nomes removidos)
- [ ] Quiz com pelo menos 6/8
- [ ] Achado em uma frase, com confiança e fontes

**Amanhã (Dia 29):** quem registrou `serraazul-pix.example`, e quando? RDAP, WHOIS e o histórico dos domínios imitação.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>